# H1: PP-OCRv6 tiny: достаточно ли минимальной CNN?

**Проверяемая гипотеза:** Exact match не ниже 95%, p95 не выше 100 мс на CPU.

LCNetV4 → reshape + проекция → CTC. Tiny обходится без отдельного attention-neck. NRTR используется как вспомогательная ветвь при обучении и не нужен при inference; применяется distillation. Это не большая авторегрессионная языковая модель.

[Теория и исследования](../research/THEORY_AND_PAPERS.md). [Зафиксированный протокол](../research/PROTOCOL_V2.md). Пороговые значения — наши инженерные критерии, не требования заказчика.

In [1]:
from pathlib import Path
import sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'captcha_lab').is_dir())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
import pandas as pd
from captcha_lab.evaluation_v2 import (manifest,run_local,run_gemini,metrics,scored,
                                      compare,all_summaries,hypothesis_results,RESULTS)
audit=json.loads((ROOT/'datasets/captcha-images/audit.json').read_text())
assert audit['holdout_images']==1984 and audit['shared_images']==200
display(pd.DataFrame([{'split':k,'n':v} for k,v in audit['source_split_counts'].items()]))
print('Holdout manifest SHA-256:',audit['holdout_manifest_sha256'])
print('Shared manifest SHA-256:',audit['shared_manifest_sha256'])
print('No prediction caches: inference cell below executes the actual model.')


,split,n
0,train,6000
1,validation,2000
2,test,2000


Holdout manifest SHA-256: 33fe043e8dec7b0e238c6bc81bb63c601699540b4b6c2025ad9480e21164f42b
Shared manifest SHA-256: 675bc9b4d93369e68817412fa08a72c6e21de23c820e00e42bc30d06c7390384
No prediction caches: inference cell below executes the actual model.


## Размеченная выборка

Публичный полный test содержит 2000 изображений; 16 просмотренных в предыдущем пилоте исключены. Метки и архивы скачаны полностью, не из preview. Функция распознавания получает только пиксели; метки присоединяются после предсказания. Исходные семь целевых меток выделены отдельно, два спорных примера исключены.

In [2]:
labels=manifest('shared_200' if 'ppocr-tiny'=='gemini' else 'holdout')
display(labels[['image_id','label','image_path']].head(12))
print('Evaluation images:',len(labels))

,image_id,label,image_path
0,test_000c1b6d4e5f11fb75d55a7c094b901e,161660,data/benchmark/images/test/161660.000c1b6d4e5f...
1,test_00105a4c053053509955d95368312987,496670,data/benchmark/images/test/496670.00105a4c0530...
2,test_001a3b40ef793b5984df8969f145d7bb,744782,data/benchmark/images/test/744782.001a3b40ef79...
3,test_0021e642491cf5bbc13cfc59fa46084a,017232,data/benchmark/images/test/017232.0021e642491c...
4,test_0047646a0e3b67ebc360e43ba88d1596,853091,data/benchmark/images/test/853091.0047646a0e3b...
5,test_0064b2188592a16991d5394b3ec5bf01,959686,data/benchmark/images/test/959686.0064b2188592...
6,test_00a5d81ab5060a0d45608169ce20bda9,159645,data/benchmark/images/test/159645.00a5d81ab506...
7,test_00e02c7c9da2cc0d3cdf5e3d9183a9f5,952376,data/benchmark/images/test/952376.00e02c7c9da2...
8,test_0123f4327a176c460aeefecd1d69bc11,374639,data/benchmark/images/test/374639.0123f4327a17...
9,test_0127972a796c1f306fdb5f919a7649fa,517493,data/benchmark/images/test/517493.0127972a796c...


Evaluation images: 1984


## Реальный прогон

Ячейка заново загружает модель и выполняет inference на каждой картинке. Ошибки сохраняются, готовые ответы не подставляются. CPU, batch=1, отдельный прогрев на validation.

In [3]:
frames=run_local('ppocr-tiny')

ppocr-tiny holdout 200 / 1984


ppocr-tiny holdout 400 / 1984


ppocr-tiny holdout 600 / 1984


ppocr-tiny holdout 800 / 1984


ppocr-tiny holdout 1000 / 1984


ppocr-tiny holdout 1200 / 1984


ppocr-tiny holdout 1400 / 1984


ppocr-tiny holdout 1600 / 1984


ppocr-tiny holdout 1800 / 1984


In [4]:
rows=[]
for dataset,predictions in frames.items():
    labels=manifest(dataset)
    rows.append(dict(dataset=dataset,**metrics(predictions,labels)))
    joined=scored(predictions,labels)
    destination=RESULTS/('ppocr-tiny')/(dataset+'_scored.csv')
    joined.to_csv(destination,index=False)
    print(dataset,'— ошибки по изображениям:')
    display(joined.loc[~joined.correct,['image_id','label','prediction','error','error_detail']].head(20))
display(pd.DataFrame(rows))


holdout — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
4,test_0047646a0e3b67ebc360e43ba88d1596,853091,.853091,,
5,test_0064b2188592a16991d5394b3ec5bf01,959686,.959686,,
9,test_0127972a796c1f306fdb5f919a7649fa,517493,.517493,,
14,test_020a007c253ce7d1c28b683e15d7e83a,443368,.443368,,
17,test_0269da701f019bbca1db3ba93e8933a5,902681,.902681,,
21,test_034e8aac2ba9b9113823c98b4907b0b1,343212,.343212,,
26,test_03c01ef06d87fc2f119aeb5117da1566,499485,499485S,,
27,test_03eb11ea68e77e842c7c1396d748e648,737065,.737065,,
28,test_03f34f710d4bbbcf8635367db15163a8,468785,488785S,,
31,test_0430cbde3b764b430c4d3beb153149ef,468669,468688,,


target — ошибки по изображениям:


,image_id,label,prediction,error,error_detail
0,33ef91d464c03eb9,894412,804L7,,
1,41ada5001f1e1e88,432077,4320m,,
3,7619f1f1b372df6d,181263,12S,,
5,d4c0652824fb2dfd,639125,62話,,
6,d4d1163a8ce99293,642044,642L,,


,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
0,holdout,1984,974,0.490927,0.468969,0.512921,0.120296,0,0,19.4651,27.02128,19.4651,27.02128
1,target,7,2,0.285714,0.082219,0.641066,0.404762,0,0,12.5152,13.28250,12.5152,13.28250


## Вывод по гипотезе

Exact match и CER вычисляются по скачанным меткам. Вердикт получается из заранее заданного критерия и интервала разности; отсутствие статистической определённости явно показывается. Правильность на публичном наборе не переносится автоматически на Instagram.

In [5]:
summary=all_summaries()
display(summary[summary.model.eq('ppocr-tiny')])
hypotheses,paired=hypothesis_results()
display(hypotheses[hypotheses.hypothesis.eq('H1')])
display(paired[paired.hypothesis.eq('H1')] if not paired.empty else paired)

,model,dataset,n,correct,exact_match,ci95_low,ci95_high,cer,errors,missing,median_ms,p95_ms,success_median_ms,success_p95_ms
0,ppocr-tiny,holdout,1984,974,0.490927,0.468969,0.512921,0.120296,0,0,19.4651,27.02128,19.4651,27.02128
1,ppocr-tiny,shared_200,200,107,0.535000,0.465866,0.602814,0.102500,0,0,19.6837,29.82381,19.6837,29.82381
2,ppocr-tiny,target,7,2,0.285714,0.082219,0.641066,0.404762,0,0,12.5152,13.28250,12.5152,13.28250


,hypothesis,criterion,verdict,estimate,ci_low,ci_high
0,H1,"tiny exact>=95%, p95<=100ms",rejected,0.490927,0.468969,0.512921


""
